In [ ]:
import logging
import math
import os
from functools import partial
from typing import Any, Literal

import matplotlib.pyplot as plt
import numpy as np
import porepy as pp
import sympy
from buckley_leverett import (
    functions,
    grid,
    misc,
    numerical_solution,
)

from tpf_lab.applications.convergence_analysis import ConvergenceAnalysisExtended
from tpf_lab.models.buckley_leverett import (
    BuckleyLeverettBoundaryConditions,
    BuckleyLeverettDataSaving,
    BuckleyLeverettDefaultGeometry,
    BuckleyLeverettEquations,
    BuckleyLeverettSemiAnalyticalSolution,
    BuckleyLeverettSetup,
    BuckleyLeverettSolutionStrategy,
    DiagnosticsMixinExtended,
    TwoPhaseFlowVariables,
    VerificationUtils,
)
from tpf_lab.numerics.ad.functions import ad_pow

# Fix seed for reproducability.
np.random.seed(0)

# Setup logging.
logger = logging.getLogger()
logger.setLevel(logging.INFO)

#### Introduction
In this notebook, we analyse the accuracy of the fractional flow formulation of the
Buckley-Leverett solution.

We compare the case linear, power and a perturbed relative permeability model.

#### Governing equations
We employ a two phase flow model in the nonwetting pressure - wetting saturation
formulation. The governing equations are
$$\begin{align*}\\
	\nabla\cdot\mathbf{u} = \nabla\cdot\left(-\lambda_{t}\nabla p_{n} + \lambda_{w}\nabla p_{c} + \lambda_{w}\nabla\rho_{w}\mathbf{g} + \lambda_{n}\nabla\rho_{n}\mathbf{g}\right) & = q_{t}, \\
	\phi\frac{\partial S_{w}}{\partial t} + \nabla\cdot\left(f_{w}\mathbf{u} + f_{w}\lambda_{n}\nabla(p_{c} + \Delta\rho\mathbf{g})\right) & = q_{w}.
\end{align*}$$
Here, $$f_w = \frac{\lambda_w}{\lambda_w + \lambda_n},\quad\lambda_l =
\frac{k_{r,l}}{\mu_l}$$ is the fractional wetting flow in dependence of the relative
phase permeabilities $k_{r,l}.$

#### Set default parameters for the model

In [ ]:
MAX_NEWTON_ITERATIONS = 30

DEFAULT_NUM_GRID_CELLS = 200
DEFAULT_PHYS_SIZE = 20
# Default grid boundaries for the BuckleyLeverett class
XMIN = -10
XMAX = 10

POROSITY = 1.0
VISCOSITY_W = 1.0
VISCOSITY_N = 1.0
DENSITY_W = 1.0
DENSITY_N = 1.0

RESIDUAL_SATURATION_W = 0.3
RESIDUAL_SATURATION_N = 0.3

REL_PERM_MODEL = "linear"
REL_PERM_LINEAR_PARAM_W = 1.0
REL_PERM_LINEAR_PARAM_N = 1.0
LIMIT_REL_PERM = True

INFLUX = 1.0
ANGLE = math.pi / 4

# This will be changed based on the CFL condition
DEFAULT_MAX_TIME_STEP: float = 0.1


# Set up folder and files for logging/plots/saved time steps.
foldername: str = os.path.join(
    "results",
    "buckley_leverett",
    "order_of_accuracy",
    "linear rel perm",
    f"NEWTON_ITERATIONS_{MAX_NEWTON_ITERATIONS}",
)
try:
    os.makedirs(foldername)
except Exception:
    pass

#### Case 1: Linear relative permeabilties
The relative phase permeabilities are given by the linear model:
$$\begin{align*}
    k_{r,w}(\hat{S}_w) & = c_1\hat{S}_w,\\
    k_{r,n}(\hat{S}_w) & = c_2(1-\hat{S}_w),
\end{align*}$$
where $\hat{S}_w$ is the normalized wetting saturation
$$\hat{S}_w=\frac{S_w-S_{w,res}}{1-S_{w,res}-S_{n,res}}.$$
We choose $c_1=c_2=1.$

To avoid the Newton solver crashing once the saturations become unphysical, the relative
permeabilties are limited from above and below
$$\begin{align*}
    k_{r,w}(\hat{S}_w) & = \min\{\max\{\hat{S}_w^m,0.01\},0.99\},\\
    k_{r,n}(\hat{S}_w) & = \min\{\max\{(1-\hat{S}_w)^m,0.01\},0.99\}.
\end{align*}$$

##### Set up analytical and Lax-Friedrichs solver
To be able to compare the numerical solution, we compute the analytical solution. Since
the total flow function is linear, each saturation has the same speed and the analytical
solution equals the initial condition.
Furthermore we set up a Lax-Friedrichs solver that can compute the CFL number.

In [ ]:
lax_friedrichs_grid = grid.create_grid(
    (XMIN, XMAX), (XMAX - XMIN) / DEFAULT_NUM_GRID_CELLS
)
initial_condition = np.full_like(lax_friedrichs_grid, RESIDUAL_SATURATION_W)
initial_condition[0 : int(DEFAULT_NUM_GRID_CELLS / 2) - 10] = 1 - RESIDUAL_SATURATION_N
initial_condition[
    int(DEFAULT_NUM_GRID_CELLS / 2) - 10 : int(DEFAULT_NUM_GRID_CELLS / 2) + 10
] = np.linspace(1 - RESIDUAL_SATURATION_N, RESIDUAL_SATURATION_W, 20)

params = {
    "formulation": "n_pressure_w_saturation",
    # grid
    "meshing_arguments": 
        {"cell_size": DEFAULT_PHYS_SIZE / float(50000)},
    "phys size": DEFAULT_PHYS_SIZE,
    # fluid and solid params
    "porosity": POROSITY,
    "viscosity_w": VISCOSITY_W,
    "viscosity_n": VISCOSITY_N,
    "density_w": DENSITY_W,
    "density_n": DENSITY_N,
    "S_M": 1 - RESIDUAL_SATURATION_W,
    "S_m": RESIDUAL_SATURATION_N,
    "residual_saturation_w": RESIDUAL_SATURATION_W,
    "residual_saturation_n": RESIDUAL_SATURATION_N,
    # rel. perm model
    "rel_perm_model": REL_PERM_MODEL,
    "rel_perm_linear_param_w": REL_PERM_LINEAR_PARAM_W,
    "rel_perm_linear_param_n": REL_PERM_LINEAR_PARAM_N,
    "limit_rel_perm": True,
    # Buckley-Leverett params
    "angle": ANGLE,
    "influx": INFLUX,
    # Lax-Friedrichs params
    "grid": lax_friedrichs_grid,
    "initial_condition": initial_condition,
    # Linear flow function. Necessary parameter for the analytical solver.
    "linear_flow": True,
}
lax_friedrichs = numerical_solution.BuckleyLeverett(params)
# Time step fulfilling the CFL condition for the default grid cells number.
DEFAULT_MAX_TIME_STEP = lax_friedrichs.cfl_condition()

We map the analytical solution:

In [ ]:
model = BuckleyLeverettSetup(params)
model.prepare_simulation()
exact_solution = model._exact_solution
fig = plt.figure()
plt.plot(model.mdg.subdomains()[0].cell_centers[0] + model.domain.bounding_box["xmin"], exact_solution, label="analytical solution")
plt.xlabel(rf"x")
plt.ylabel(rf"S_w")
plt.legend()
fig.subplots_adjust(left=0., bottom=0.1)
plt.savefig(os.path.join(foldername, "analytical_solution.png"))
plt.close()
misc.map_fractional_flow(
    model.analytical,
    filename=os.path.join(foldername, "analytical_solution"),
)

##### Convergence analysis in time

In [ ]:
def save_convergence_results(
        analysis: ConvergenceAnalysisExtended,
        results: list,
        level_type: Literal["time", "space"]="time"
    ) -> None:
    x_axis = "time_step" if level_type == "time" else "cell_diameter"
    ooc = analysis.order_of_convergence(
        analysis.transform_results_to_classical(results),
        x_axis=x_axis
        )
    logger.info(f"Order of convergence: {ooc}")
    with open(os.path.join(foldername, f"order_of_convergence_in_{level_type}.txt"), "w") as f:
        f.write(f"order of convergence: {ooc}")

    final_errors, cell_diameter_levels , time_step_levels = analysis.final_l2_error(results)
    xx = time_step_levels if level_type == "time" else cell_diameter_levels
    # Plot final error for each parameter.
    fig = plt.figure()
    plt.plot(
        xx,
        final_errors,
        "xb-",
        label=f"default power rel. perm ({MAX_NEWTON_ITERATIONS} iter)",
    )
    plt.xlabel(r"\log_{10}(\Delta t)")
    plt.ylabel(r"\log_{10}(\|e\|)")
    plt.xscale("log")
    plt.yscale("log")
    plt.title("Convergence Analysis")
    plt.legend()
    fig.subplots_adjust(left=0., bottom=0.1)
    plt.savefig(os.path.join(foldername, f"accuracy_in_{level_type}.png"))

    avg_nonlinear_iterations, cell_diameter_levels , time_step_levels = analysis.average_number_of_iterations(results)
    xx = time_step_levels if level_type == "time" else cell_diameter_levels
    fig = plt.figure()
    plt.plot(
        xx,
        avg_nonlinear_iterations,
        "xb-",
        label=f"default power rel. perm",
    )
    plt.xlabel(r"\log_{10}(\Delta t)")
    plt.ylabel(r"n_{iterations}")
    plt.xscale("log")
    plt.title("average Newton iterations")
    plt.legend()
    fig.subplots_adjust(left=0.1, bottom=0.1)
    plt.savefig(
        os.path.join(foldername, f"average_newton_iterations_varying_{x_axis}.png")
    )

In [ ]:
params.update(
    {
        "folder_name": foldername,
        "meshing_arguments": 
            {"cell_size": DEFAULT_PHYS_SIZE / float(DEFAULT_NUM_GRID_CELLS)},
        "time_manager": pp.TimeManager(
            schedule=np.array([0, 1]),
            dt_init=0.1,
            constant_dt=True,
            ),
        "limit_rel_perm": True,
    }
)

analysis = ConvergenceAnalysisExtended(BuckleyLeverettSetup, params, levels=7, temporal_refinement_rate=2)
results = analysis.run_analysis()
analysis.export_results_to_json(
    results,
    variables_to_export=["l2_error", "residuals", "iteration_counter", "time"],
    file_name=os.path.join(foldername,"temporal_error_analysis.json")
)
save_convergence_results(analysis, results, "time")

##### Convergence analysis in space

In [ ]:
params.update({
        "meshing_arguments": 
            {"cell_size": DEFAULT_PHYS_SIZE / 50.0},
    "time_step_size": DEFAULT_MAX_TIME_STEP,
})

analysis = ConvergenceAnalysisExtended(BuckleyLeverettSetup, params, levels=6, spatial_refinement_rate=2)
results = analysis.run_analysis()
analysis.export_results_to_json(
    results,
    variables_to_export=["l2_error", "residuals", "iteration_counter", "time"],
    file_name=os.path.join(foldername,"spatial_error_analysis.json")
)

save_convergence_results(analysis, results, "space")

#### Case 2: Corey relative permeabilties
The relative phase permeabilities are given by the Corey model:
$$\begin{align*}
    k_{r,w}(\hat{S}_w) & = \hat{S}_w^m,\\
    k_{r,n}(\hat{S}_w) & = (1-\hat{S}_w)^m,
\end{align*}$$
where $\hat{S}_w$ is the normalized wetting saturation
$$\hat{S}_w=\frac{S_w-S_{w,res}}{1-S_{w,res}-S_{n,res}}.$$
We choose $m=3.$

Again, the relative permeabilities are limited above and below.

We change the model parameters,...

In [ ]:
REL_PERM_MODEL = "power"
params.update({"rel_perm_model": REL_PERM_MODEL, "linear_flow": False})

# Set up folder and files for logging/plots/saved time steps.
foldername = os.path.join(
    "results",
    "buckley_leverett",
    "order_of_accuracy",
    "Corey_rel_perm",
    f"NEWTON_ITERATIONS_{MAX_NEWTON_ITERATIONS}",
)
try:
    os.makedirs(foldername)
except Exception:
    pass

...reinitialize the Lax-Friedrichs solver to obtain the CFL condition,...

In [ ]:
lax_friedrichs = numerical_solution.BuckleyLeverett(params)
# Time step fulfilling the CFL condition for the default grid cells number.
DEFAULT_MAX_TIME_STEP = lax_friedrichs.cfl_condition()

...map the analytical solution,...

In [ ]:
model = BuckleyLeverettSetup(params)
model.prepare_simulation()
exact_solution = model._exact_solution
fig = plt.figure()
plt.plot(model.mdg.subdomains()[0].cell_centers[0] + model.domain.bounding_box["xmin"], exact_solution, label="analytical solution")
plt.xlabel(rf"x")
plt.ylabel(rf"S_w")
plt.legend()
fig.subplots_adjust(left=0.1, bottom=0.1)
plt.savefig(os.path.join(foldername, "analytical_solution.png"))
plt.close()
misc.map_fractional_flow(
    model.analytical,
    filename=os.path.join(foldername, "analytical_solution"),
)

...before we let the model run again.

##### Convergence analysis in time

In [ ]:
params.update(
    {
        "folder_name": foldername,
        "meshing_arguments": 
            {"cell_size": DEFAULT_PHYS_SIZE / float(DEFAULT_NUM_GRID_CELLS)},
        "time_manager": pp.TimeManager(
            schedule=np.array([0, 1]),
            dt_init=0.1,
            constant_dt=True,
            ),
    }
)

analysis = ConvergenceAnalysisExtended(BuckleyLeverettSetup, params, levels=7, temporal_refinement_rate=2)
results = analysis.run_analysis()
analysis.export_results_to_json(
    results,
    variables_to_export=["l2_error", "residuals", "iteration_counter", "time"],
    file_name=os.path.join(foldername,"temporal_error_analysis.json")
)
save_convergence_results(analysis, results, "time")

##### Convergence analysis in space

In [ ]:
params.update(
    {
        "meshing_arguments": 
            {"cell_size": DEFAULT_PHYS_SIZE / 50.0},
        "time_step_size": DEFAULT_MAX_TIME_STEP,
    }
)

analysis = ConvergenceAnalysisExtended(BuckleyLeverettSetup, params, levels=6, spatial_refinement_rate=2)
results = analysis.run_analysis()
analysis.export_results_to_json(
    results,
    variables_to_export=["l2_error", "residuals", "iteration_counter", "time"],
    file_name=os.path.join(foldername,"spatial_error_analysis.json")
)

save_convergence_results(analysis, results, "space")

#### Case 3: Perturbated wetting mobility
We add an some noise to the wetting relative permeability.
$$\tilde{k}_{r,w}(\hat{S}_w)$$

We adjust the flow function for the analytical solution and the fractional flow
formulation accordingly.

In [ ]:
class BuckleyLeverettEquations_WobblyRelPerm(BuckleyLeverettEquations):
    _yscales: list[float]
    _xscales: list[float]
    _offsets: list[float]

    def _mobility_w(self, subdomains: list[pp.Grid]) -> pp.ad.Operator:
        """Add a perturbation to the wetting mobility."""
        mobility_w = super()._mobility_w(subdomains)
        upwind_w = pp.ad.UpwindAd(self.w_flux_key, subdomains)
        return mobility_w + upwind_w.upwind @ self._error_function_deriv()

    def _error_function_deriv(self) -> pp.ad.Operator:
        """Returns the derivative of the error function w.r.t. the saturation.

        This can be used to simulate perturbations in the cap. pressure and rel. perm.
        models.

        Returns:
            Derivative of the error function in terms of :math:`S_w`.
        """
        s = self._ad.saturation
        xscales = [pp.ad.Scalar(xscale) for xscale in self._xscales]
        yscales = [pp.ad.Scalar(yscale) for yscale in self._yscales]
        offsets = [pp.ad.Scalar(offset) for offset in self._offsets]
        exp_func = pp.ad.Function(pp.ad.functions.exp, "exp")
        square_func = pp.ad.Function(partial(ad_pow, exponent=2), "square")
        error = pp.ad.Scalar(0) * s
        for xscale, yscale, offset in zip(xscales, yscales, offsets):
            error = error + yscale * exp_func(
                pp.ad.Scalar(-1) * xscale * square_func(s - offset)
            )
        return error

class BuckleyLeverettSolutionStrategy_WobblyRelPerm(BuckleyLeverettSolutionStrategy):
    def __init__(self, params: dict | None = None) -> None:
        super().__init__(params)
        if params is None:
            params = {}
        # Parameters for the error function derivative:
        self._yscales: list[float] = params.get("yscales", [1.0])
        self._xscales: list[float] = params.get("xscales", [200])
        self._offsets: list[float] = params.get("offsets", [0.5])


class BuckleyLeverettSetup_WobblyRelPerm(  # type: ignore
    BuckleyLeverettEquations_WobblyRelPerm,
    TwoPhaseFlowVariables,
    BuckleyLeverettBoundaryConditions,
    BuckleyLeverettSolutionStrategy_WobblyRelPerm,
    #
    BuckleyLeverettDefaultGeometry,
    #
    BuckleyLeverettSemiAnalyticalSolution,
    BuckleyLeverettDataSaving,
    VerificationUtils,
    DiagnosticsMixinExtended,
):
    ...


class WobblyFractionalFlowSympy(functions.FractionalFlowSymPy):
    def __init__(self, params: dict[str, Any]) -> None:
        super().__init__(params)
        # Parameters for the error function derivative:
        self.yscales: list[float] = params.get("yscales", [1.0])
        self.xscales: list[float] = params.get("xscales", [200])
        self.offsets: list[float] = params.get("offsets", [0.5])

    def lambda_w(self):
        r"""Wetting phase mobility.

        Power model
        .. math::
            k_{r,w}(S_w)=S_w^3 + \epsilon(S_w)

        """
        return self.S_normalized() ** 3 + self.error_function_deriv()

    def error_function_deriv(self):
        return sympy.Add(
            *[
                yscale * sympy.exp(-xscale * (self.S_w - offset) ** 2)
                for xscale, yscale, offset in zip(
                    self.xscales, self.yscales, self.offsets
                )
            ]
        )

Next, we set up the parameters for the perturbed flow function and the save location,...

In [ ]:
# Parameters for wobbly rel. perm.
YSCALES = np.maximum(np.random.rand(20), 0.5).tolist()
YSCALES[:5] = np.arange(0.1, 0.5, 0.1)
XSCALES = [20000.0] * 20
OFFSETS = np.linspace(0.4, 0.6, 20).tolist()
params.update(
    {
    "yscales": YSCALES,
    "xscales": XSCALES,
    "offsets": OFFSETS,
    }
)

# Set up folder and files for logging/plots/saved time steps.
foldername = os.path.join(
    "results",
    "buckley_leverett",
    "order_of_accuracy",
    "wobbly_rel_perm",
    f"NEWTON_ITERATIONS_{MAX_NEWTON_ITERATIONS}",
)
try:
    os.makedirs(foldername)
except Exception:
    pass

...reinitialize the Lax-Friedrichs solver to obtain the CFL condition,...

In [ ]:
lax_friedrichs = numerical_solution.BuckleyLeverett(params)
lax_friedrichs.fractionalflow = WobblyFractionalFlowSympy(params)
lax_friedrichs.lambdify()
# Time step fulfilling the CFL condition for the default grid cells number.
DEFAULT_MAX_TIME_STEP = lax_friedrichs.cfl_condition()

...and recalculate the analytical solution.

In [ ]:
model = BuckleyLeverettSetup_WobblyRelPerm(params)
model.analytical.fractionalflow = WobblyFractionalFlowSympy(params)
model.analytical.lambdify()

model.prepare_simulation()
exact_solution = model._exact_solution
fig = plt.figure()
plt.plot(model.mdg.subdomains()[0].cell_centers[0] + model.domain.bounding_box["xmin"], exact_solution, label="analytical solution")
plt.xlabel(rf"x")
plt.ylabel(rf"S_w")
plt.legend()
fig.subplots_adjust(left=0., bottom=0.1)
plt.savefig(os.path.join(foldername, "analytical_solution.png"))
plt.close()
misc.map_fractional_flow(
    model.analytical,
    filename=os.path.join(foldername, "analytical_solution"),
)

##### Convergence analysis in time

In [ ]:
params.update(
    {
        "folder_name": foldername,
        "meshing_arguments": 
            {"cell_size": DEFAULT_PHYS_SIZE / float(DEFAULT_NUM_GRID_CELLS)},
        "time_manager": pp.TimeManager(
            schedule=np.array([0, 1]),
            dt_init=0.1,
            constant_dt=True,
            ),
    }
)

analysis = ConvergenceAnalysisExtended(BuckleyLeverettSetup_WobblyRelPerm, params, levels=7, temporal_refinement_rate=2)
results = analysis.run_analysis()
analysis.export_results_to_json(
    results,
    variables_to_export=["l2_error", "residuals", "iteration_counter", "time"],
    file_name=os.path.join(foldername,"temporal_error_analysis.json")
)
save_convergence_results(analysis, results, "time")

##### Convergence analysis in space

In [ ]:
params.update(
    {
        "meshing_arguments": 
            {"cell_size": DEFAULT_PHYS_SIZE / 50.0},
        "time_step_size": DEFAULT_MAX_TIME_STEP,
    }
)

analysis = ConvergenceAnalysisExtended(BuckleyLeverettSetup_WobblyRelPerm, params, levels= 6, spatial_refinement_rate=2)
results = analysis.run_analysis()
analysis.export_errors_to_txt(results, os.path.join(foldername,"spatial_error_analysis.txt"))
ooc = analysis.order_of_convergence(results, [])

save_convergence_results(analysis, results, "space")